In [1]:
import pandas as pd


In [2]:
print('hello')

hello


In [3]:
title = pd.read_csv('./data/truncated_title.basics.tsv',sep='\t')

In [4]:
ratings = pd.read_csv('./data/truncated_title.ratings.tsv',sep='\t')

In [5]:
title.rename(columns={'tconst': 'title_id'}, inplace=True)
title.rename(columns={'primaryTitle': 'title'}, inplace=True)
ratings.rename(columns={'tconst': 'title_id'}, inplace=True)

In [6]:
title

,title_id,titleType,title,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000929,short,Klebolin klebt alles,Klebolin klebt alles,0,1990,\N,\N,"Comedy,Short"
1,tt0000977,short,Mutterliebe,Mutterliebe,0,1990,\N,\N,Short
2,tt0034841,short,Hen Hop,Hen Hop,0,1994,\N,4,"Animation,Short"
3,tt0040844,short,Crossroads of Laredo,Crossroads of Laredo,0,1995,\N,30,"Short,Western"
4,tt0078006,short,Norman and the Killer,Norman and the Killer,0,1991,\N,30,"Horror,Short"
...,...,...,...,...,...,...,...,...,...
495,tt0103054,tvEpisode,Wer zweimal stirbt,Wer zweimal stirbt,0,1991,\N,88,"Crime,Drama,Mystery"
496,tt0103062,tvEpisode,Tell Me That You Love Me,Tell Me That You Love Me,0,1991,\N,98,"Comedy,Crime,Drama"
497,tt0103071,tvEpisode,Thanners neuer Job,Thanners neuer Job,0,1991,\N,83,"Crime,Drama,Mystery"
498,tt0103145,tvEpisode,Two Teens and a Baby,Two Teens and a Baby,0,1992,\N,\N,"Adventure,Comedy,Drama"


In [7]:
ratings

,title_id,averageRating,numVotes
0,tt0000929,5.3,46
1,tt0015414,5.2,16
2,tt0015724,6.1,26
3,tt0034841,6.3,282
4,tt0038086,7.0,27
...,...,...,...
437,tt0103054,5.2,50
438,tt0103062,8.2,41
439,tt0103071,6.6,56
440,tt0103145,7.6,12


In [8]:
ratings.isnull().sum()

title_id         0
averageRating    0
numVotes         0
dtype: int64

In [9]:
title[title['title'] == 'Rejsen mod en fødsel']

,title_id,titleType,title,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
32,tt0098182,short,Rejsen mod en fødsel,Rejsen mod en fødsel,0,1990,\N,8,"Animation,Short"


In [10]:
ratings[ratings['title_id'] == 'tt0098182']

,title_id,averageRating,numVotes


In [11]:
new_title = title[['title_id' ,'title','genres']]

In [12]:
new_title

,title_id,title,genres
0,tt0000929,Klebolin klebt alles,"Comedy,Short"
1,tt0000977,Mutterliebe,Short
2,tt0034841,Hen Hop,"Animation,Short"
3,tt0040844,Crossroads of Laredo,"Short,Western"
4,tt0078006,Norman and the Killer,"Horror,Short"
...,...,...,...
495,tt0103054,Wer zweimal stirbt,"Crime,Drama,Mystery"
496,tt0103062,Tell Me That You Love Me,"Comedy,Crime,Drama"
497,tt0103071,Thanners neuer Job,"Crime,Drama,Mystery"
498,tt0103145,Two Teens and a Baby,"Adventure,Comedy,Drama"


In [13]:
from sklearn.feature_extraction.text import CountVectorizer

In [14]:
cv=CountVectorizer(max_features=500, stop_words='english')

In [15]:
cv

CountVectorizer(max_features=500, stop_words='english')

In [16]:
vector=cv.fit_transform(new_title['genres'].values.astype('U')).toarray()

In [17]:
vector.shape

(500, 25)

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

In [19]:
similarity=cosine_similarity(vector)

In [20]:
similarity

array([[1.        , 0.70710678, 0.5       , ..., 0.        , 0.40824829,
        0.        ],
       [0.70710678, 1.        , 0.70710678, ..., 0.        , 0.        ,
        0.        ],
       [0.5       , 0.70710678, 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.33333333,
        1.        ],
       [0.40824829, 0.        , 0.        , ..., 0.33333333, 1.        ,
        0.33333333],
       [0.        , 0.        , 0.        , ..., 1.        , 0.33333333,
        1.        ]])

In [21]:
import pickle

In [22]:
pickle.dump(new_title, open('movies_list.pkl', 'wb'))

In [23]:
pickle.dump(ratings, open('ratings.pkl', 'wb'))

In [24]:
pickle.dump(similarity, open('similarity.pkl', 'wb'))

In [25]:
new_title[new_title['title']=='Hen Hop'].index[0]

2

In [26]:
def recommend(movie):
    index = int(new_title[new_title['title'] == movie].index[0])
    distance = sorted(list(enumerate(similarity[index])), reverse=True, key=lambda vector: vector[1])
    j = 0
    counter = 0
    recommend_movies = []  
    while counter <= 10 and j <= 500:
        i = distance[j]
        j = j+1
        title_id = new_title.iloc[i[0]].title_id
        avg_rating_series = ratings[ratings['title_id'] == title_id]['averageRating']

        if avg_rating_series.empty:
            continue
        counter += 1
        # Extract a scalar value from the Series or set avg_rating to NaN
        avg_rating = avg_rating_series.iloc[0] 

        recommend_movies.append({
            'title': new_title.iloc[i[0]].title,
            'avg_rating': avg_rating
        })

    # Sort recommend_movies based on avg_rating
    sorted_movies = sorted(recommend_movies, key=lambda x: x['avg_rating'], reverse=True)

    return sorted_movies


In [27]:
recommend('Hen Hop')

[{'title': 'The Making of Monsters', 'avg_rating': 8.4},
 {'title': 'Next', 'avg_rating': 7.4},
 {'title': 'That Burning Question', 'avg_rating': 7.0},
 {'title': 'Western', 'avg_rating': 6.8},
 {'title': 'The Hill Farm', 'avg_rating': 6.7},
 {'title': 'Feet of Song', 'avg_rating': 6.7},
 {'title': 'God morgon Gerda Gök', 'avg_rating': 6.7},
 {'title': 'Ident', 'avg_rating': 6.5},
 {'title': 'Hen Hop', 'avg_rating': 6.3},
 {'title': 'Zwisch!', 'avg_rating': 6.2},
 {'title': 'Die Beichte', 'avg_rating': 5.2}]